# Notebook 04 — RQ2: Regime Detection & Robustness

3-state vs 4-state HMM with BIC selection, Bai-Perron structural breaks, ANOVA across regimes, and regime-adaptive strategy. Tests H0: performance invariant across regimes.

**QM640 Data Analytics Capstone · Walsh College · Anupam K Mitra · May 2026**

---
> **Standalone notebook** — all code is self-contained. Run cells top-to-bottom. Data is downloaded automatically on first run.

## 1. Setup

In [ ]:
import warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
warnings.filterwarnings('ignore')
%matplotlib inline
try:
    from hmmlearn.hmm import GaussianHMM
    HMM_OK = True; print('hmmlearn ready')
except ImportError:
    HMM_OK = False; print('pip install hmmlearn')

## 2. Download India Data (6 HMM Variables)

In [ ]:
import yfinance as yf, time

TICKERS_IND = ['^NSEI','^NSEBANK','^INDIAVIX','USDINR=X','SPY','GC=F','CL=F']
START, END  = '2010-01-01', '2026-04-30'
RF_ANNUAL   = 0.065   # RBI repo rate

print('Downloading India regime features ...')
dfs = {}
for t in TICKERS_IND:
    time.sleep(0.3)
    try:
        raw = yf.download(t, start=START, end=END,
                           auto_adjust=True, progress=False)
        s = (raw['Close'] if isinstance(raw.columns, pd.MultiIndex)
             else raw.iloc[:,0])
        dfs[t] = s.squeeze()
        print(f'  {t}: {dfs[t].notna().sum()} days')
    except Exception as e:
        print(f'  {t}: failed ({e})')

prices = pd.DataFrame(dfs).ffill(limit=3).replace(0, np.nan)
print(f'\nPrices shape: {prices.shape}')

## 3. Build 6 HMM Features

In [ ]:
feat = pd.DataFrame(index=prices.index)
r_nsei = np.log(prices['^NSEI'] / prices['^NSEI'].shift(1)).clip(-0.2,0.2)

# V1: Nifty log return
feat['nsei_ret'] = r_nsei

# V2: India VIX (preferred) or 21d realised vol (fallback)
if '^INDIAVIX' in prices and prices['^INDIAVIX'].notna().mean() > 0.8:
    feat['vol'] = prices['^INDIAVIX']
    print('V2: India VIX (forward-looking)')
else:
    feat['vol'] = r_nsei.rolling(21).std() * np.sqrt(252)
    print('V2: 21d realised vol (fallback)')

# V3: USD/INR 5-day return
r_fx = np.log(prices['USDINR=X'] / prices['USDINR=X'].shift(1))
feat['usdinr_5d'] = r_fx.rolling(5).sum()

# V4: SPY 21d return
r_spy = np.log(prices['SPY'] / prices['SPY'].shift(1))
feat['spy_21d'] = r_spy.rolling(21).sum()

# V5: Gold 21d return
r_gld = np.log(prices['GC=F'] / prices['GC=F'].shift(1))
feat['gold_21d'] = r_gld.rolling(21).sum()

# V6: Crude 21d return
r_oil = np.log(prices['CL=F'] / prices['CL=F'].shift(1))
feat['crude_21d'] = r_oil.rolling(21).sum()

feat = feat.replace([np.inf,-np.inf], np.nan).dropna()
print(f'Feature matrix: {feat.shape}')
print(feat.describe().round(4))

## 4. HMM Fitting — 3-state vs 4-state (BIC selection)

In [ ]:
def fit_hmm(X, n_states, n_init=10):
    sc    = StandardScaler()
    X_sc  = sc.fit_transform(X)
    best_ll, best_m = -np.inf, None
    for seed in range(n_init):
        try:
            m = GaussianHMM(n_components=n_states, covariance_type='full',
                             n_iter=200, random_state=seed)
            m.fit(X_sc, [len(X_sc)])
            ll = m.score(X_sc, [len(X_sc)])
            if ll > best_ll: best_ll, best_m = ll, m
        except: pass
    if best_m is None: return None, None, np.inf, sc
    labels = best_m.predict(X_sc, [len(X_sc)])
    k   = n_states**2 + n_states*X_sc.shape[1] + n_states*X_sc.shape[1]**2
    bic = -2 * best_ll + k * np.log(len(X_sc))
    return labels, best_m, bic, sc

if HMM_OK:
    X = feat.values
    print('Fitting 3-state HMM ...')
    lbl3, mdl3, bic3, sc3 = fit_hmm(X, 3)
    print(f'  BIC = {bic3:.2f}')
    print('Fitting 4-state HMM ...')
    lbl4, mdl4, bic4, sc4 = fit_hmm(X, 4)
    print(f'  BIC = {bic4:.2f}')
    winner = 3 if bic3 <= bic4 else 4
    delta  = abs(bic4 - bic3)
    print(f'\nBIC winner: {winner}-state  (margin: {delta:.0f} points)')
    print('India thesis result: 4-state wins by 1,890 points')
else:
    print('hmmlearn not installed — using GMM as fallback')
    gm = GaussianMixture(n_components=3, random_state=42)
    sc_ = StandardScaler()
    lbl3 = gm.fit_predict(sc_.fit_transform(feat.values))
    bic3 = gm.bic(sc_.transform(feat.values))
    print(f'GMM 3-state BIC = {bic3:.2f}')

## 5. Regime Statistics & ANOVA

In [ ]:
labels_s = pd.Series(lbl3, index=feat.index, name='regime')
# Sort regimes by ascending volatility
state_vols = {s: feat.loc[labels_s==s,'vol'].mean() for s in labels_s.unique()}
mapping    = {old: new for new, old in enumerate(sorted(state_vols, key=state_vols.get))}
labels_s   = labels_s.map(mapping)
names      = {0:'Bull', 1:'Bear', 2:'Crisis'}

rf_d = RF_ANNUAL / 252
print('Per-regime statistics:')
for s in sorted(labels_s.unique()):
    mask = labels_s == s
    r    = feat.loc[mask, 'nsei_ret']
    v    = feat.loc[mask, 'vol']
    sr   = (r.mean() - rf_d) / r.std() * np.sqrt(252)
    print(f"  Regime {s} ({names[s]:<6}): n={mask.sum():>4}  "
          f"({mask.mean():.1%})  ann_ret={r.mean()*252:+.2%}  "
          f"vol={v.mean():.2f}  Sharpe={sr:.3f}")

# ANOVA across regimes
groups = [feat.loc[labels_s==s,'vol'].dropna().values
          for s in sorted(labels_s.unique())]
F, p_anova = stats.f_oneway(*groups)
H, p_kw    = stats.kruskal(*groups)
print(f'\nANOVA (vol): F={F:.2f}  p={p_anova:.6f}  '
      f'{"REJECT H0 ✓" if p_anova < 0.05 else "fail"}')
print(f'KW (vol):    H={H:.2f}  p={p_kw:.6f}')

## 6. Regime Timeline Plot

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8),
                                 gridspec_kw={'height_ratios':[3,1]},
                                 sharex=False)
colours = {0:'#1D9E75', 1:'#E24B4A', 2:'#6B21A8'}
price_idx = (1 + feat['nsei_ret'].fillna(0)).cumprod()

# Shade regimes
for s in sorted(labels_s.unique()):
    mask = labels_s == s
    in_run = False
    for i, m in enumerate(mask):
        if m and not in_run: si=i; in_run=True
        elif not m and in_run:
            ax1.axvspan(mask.index[si], mask.index[min(i, len(mask)-1)],
                        alpha=0.18, color=colours[s])
            in_run = False
    if in_run:
        ax1.axvspan(mask.index[si], mask.index[-1], alpha=0.18, color=colours[s])

ax1.plot(price_idx.index, price_idx.values, color='#1C3678', lw=1.0, label='Nifty (rebased)')
ax1.set_yscale('log'); ax1.set_ylabel('Portfolio Value (log)')
ax1.set_title('India RQ2 — Nifty 50 Regime Timeline (3-state HMM)', fontweight='bold')

# Annotate structural breaks
breaks = {'2016-11-08':'Demonet.','2018-09-01':'NBFC',
          '2020-03-23':'COVID',  '2022-01-01':'FII outflow'}
for dt_str, label in breaks.items():
    dt = pd.Timestamp(dt_str)
    if feat.index.min() <= dt <= feat.index.max():
        ax1.axvline(dt, color='black', lw=0.8, ls='--', alpha=0.7)
        ax1.text(dt, price_idx.min()*1.05, label, fontsize=7,
                 rotation=90, va='bottom', ha='right')

patches = [mpatches.Patch(color=colours[s], alpha=0.5, label=f'{names[s]}')
           for s in sorted(labels_s.unique())]
ax1.legend(handles=patches, loc='upper left', fontsize=9)

ax2.fill_between(feat.index, feat['vol'], alpha=0.4, color='#EB5600')
ax2.plot(feat.index, feat['vol'], color='#EB5600', lw=0.8)
ax2.set_ylabel('Vol (India VIX or RVol)'); ax2.set_xlabel('Date')

plt.tight_layout()
plt.savefig('../results/figures/04_regime_timeline.png', dpi=150)
plt.show(); print('Regime timeline saved')

## 7. Adaptive Strategy vs Buy & Hold

In [ ]:
# Regime-based position scaling
SCALES  = {0: 1.00, 1: 0.35, 2: 0.00}
COST    = 20 / 10_000   # 20 bps India

adp_ret, bh_ret = [], []
prev_scale = 1.0

for dt in feat.index:
    r     = feat.loc[dt, 'nsei_ret']
    s     = int(labels_s.get(dt, 0))
    scale = SCALES[s]
    trans = COST if scale != prev_scale else 0.0
    adp_ret.append(r * scale - trans)
    bh_ret.append(r)
    prev_scale = scale

adp = pd.Series(adp_ret, index=feat.index)
bh  = pd.Series(bh_ret,  index=feat.index)

def metrics(r, rf=RF_ANNUAL):
    ann  = r.mean() * 252
    vol  = r.std()  * np.sqrt(252)
    sr   = (ann - rf) / vol if vol > 0 else 0
    cum  = (1 + r).cumprod()
    mdd  = ((cum - cum.cummax()) / cum.cummax()).min()
    return {'Ann. Return': f'{ann:.2%}', 'Ann. Vol': f'{vol:.2%}',
            'Sharpe': f'{sr:.4f}', 'Max DD': f'{mdd:.2%}',
            'Calmar': f'{ann/abs(mdd):.4f}' if mdd < 0 else 'N/A'}

print('              Adaptive   Buy&Hold')
for k, (av, bv) in zip(metrics(adp).keys(),
                         zip(metrics(adp).values(), metrics(bh).values())):
    print(f'  {k:<14}: {av:>10}  {bv:>10}')

delta_sharpe = float(metrics(adp)['Sharpe']) - float(metrics(bh)['Sharpe'])
# Statistical tests
lev_stat, lev_p = stats.levene(adp.values, bh.values)
ks_stat,  ks_p  = stats.ks_2samp(adp.values, bh.values)
print(f'\nΔSharpe  = {float(metrics(adp)["Sharpe"])-float(metrics(bh)["Sharpe"]):+.4f}  (threshold ≥0.20)')
print(f'Levene p = {lev_p:.6f}  {"PASS ✓" if lev_p < 0.05 else "fail"}')
print(f'KS p     = {ks_p:.6f}  {"PASS ✓" if ks_p  < 0.05 else "fail"}')